In [ ]:
# ==============================================================
# FUNDAMENTOS / ANÁLISIS DE DATOS — versión alineada con la referencia
# ==============================================================

# 0. INSTALACIÓN (solo en Colab)
!pip install -q plotly python-dotenv

# ==============================================================
# 1. IMPORTACIONES
# ==============================================================
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import requests
from dotenv import load_dotenv

# ==============================================================
# 2. CONFIGURACIÓN (.env)
# ==============================================================
# Crea el .env solo si no existe (útil en Colab).
if not Path(".env").exists():
    Path(".env").write_text(
        "API_URL=https://data-charts-api.hexlet.app\n"
        "DATE_BEGIN=2023-03-01\n"
        "DATE_END=2023-09-01\n"
    )

# Sin override: si el test define variables de entorno, se respetan.
load_dotenv()

DATE_BEGIN = os.getenv("DATE_BEGIN")
DATE_END = os.getenv("DATE_END")
API_URL = os.getenv("API_URL")

if not all([API_URL, DATE_BEGIN, DATE_END]):
    raise ValueError(
        f"Faltan variables: API_URL={API_URL}, "
        f"DATE_BEGIN={DATE_BEGIN}, DATE_END={DATE_END}"
    )

Path("charts").mkdir(exist_ok=True)
sns.set(style="whitegrid")

# ==============================================================
# 3. ads.csv (descarga solo si no está en la carpeta)
# ==============================================================
if not Path("ads.csv").exists():
    url = "https://drive.google.com/uc?export=download&id=12vCtGhJlcK_CBcs8ES3BfEPbk6OJ45Qj"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    Path("ads.csv").write_bytes(r.content)

# ==============================================================
# 4. LIMPIEZA (igual que la referencia)
# ==============================================================
def clean_registration_data(df):
    df = df.drop_duplicates().copy()
    df["platform"] = df["platform"].fillna("web")
    return df


def clean_visit_data(df):
    df = df.copy()
    print("TOTAL_VISITS =", len(df))

    df["visit_dt"] = pd.to_datetime(df["visit_dt"])

    # 1) Quitar bots PRIMERO (antes de deduplicar)
    df = df[~df["user_agent"].str.contains("bot", case=False, na=False)]

    # 2) Última visita por usuario (después de quitar bots)
    df = (
        df.sort_values(by=["anonymous_id", "visit_dt"])
          .drop_duplicates(subset="anonymous_id", keep="last")
    )

    # 3) Quitar comillas invertidas alrededor de platform
    df["platform"] = df["platform"].str.strip("`")

    return df


def clean_ads_data(df):
    df = df.drop_duplicates().copy()
    df["date"] = pd.to_datetime(df["date"])
    return df

# ==============================================================
# 5. AGRUPACIÓN POR FECHA
# ==============================================================
def _assign_date_group(series, date_group):
    freq = {"day": "D", "week": "W", "month": "M", "quarter": "Q"}
    if date_group not in freq:
        raise ValueError(f"date_group inválido: {date_group}")
    return series.dt.to_period(freq[date_group]).dt.start_time.dt.date


def filter_and_aggregate_registration_data(df, date_start, date_end, date_group):
    date_start = pd.to_datetime(date_start)
    date_end = pd.to_datetime(date_end)

    if "registration_dt" not in df.columns:
        raise ValueError("'registration_dt' no existe en el DataFrame")

    df = df.copy()
    df["registration_dt"] = pd.to_datetime(df["registration_dt"], errors="coerce")

    mask = (df["registration_dt"] >= date_start) & (df["registration_dt"] <= date_end)
    filtered = df.loc[mask].copy()

    if filtered.empty:
        print("No hay registros en el rango dado")
        return pd.DataFrame(), pd.DataFrame()

    filtered["date_group"] = _assign_date_group(filtered["registration_dt"], date_group)

    with_type = (
        filtered.groupby(["date_group", "platform", "registration_type"])
        .size().reset_index(name="registrations")
        .sort_values(by="date_group")
    )
    without_type = (
        filtered.groupby(["date_group", "platform"])
        .size().reset_index(name="registrations")
        .sort_values(by="date_group")
    )
    return with_type, without_type


def filter_and_aggregate_visit_data(df, date_start, date_end, date_group):
    date_start = pd.to_datetime(date_start)
    date_end = pd.to_datetime(date_end)

    mask = (df["visit_dt"] >= date_start) & (df["visit_dt"] <= date_end)
    filtered = df.loc[mask].copy()

    if filtered.empty:
        print("No hay visitas en el rango dado")
        return pd.DataFrame()

    filtered["date_group"] = _assign_date_group(filtered["visit_dt"], date_group)

    return (
        filtered.groupby(["date_group", "platform"])
        .size().reset_index(name="visits")
    )


def filter_and_aggregate_ads_data(df, date_start, date_end, date_group):
    date_start = pd.to_datetime(date_start)
    date_end = pd.to_datetime(date_end)

    filtered = df[(df["date"] >= date_start) & (df["date"] <= date_end)].copy()

    if filtered.empty:
        print("No hay costos de publicidad en el rango dado")
        return pd.DataFrame(), pd.DataFrame()

    filtered["date_group"] = _assign_date_group(filtered["date"], date_group)
    filtered["full_ad"] = (
        filtered["source"] + " " + filtered["medium"] + " " + filtered["campaign"]
    )

    aggregated = filtered.groupby(["date_group"]).sum(numeric_only=True).reset_index()
    by_ad = filtered.groupby(["date_group", "full_ad"]).sum(numeric_only=True).reset_index()
    return aggregated, by_ad

# ==============================================================
# 6. CONVERSIÓN (igual que la referencia)
# ==============================================================
def merge_dataframes_and_calculate_conversion(df_visits, df_registrations):
    # Inner join: solo combinaciones fecha+plataforma presentes en ambos
    merged = pd.merge(df_visits, df_registrations, on=["date_group", "platform"])
    merged["conversion"] = merged["registrations"] / merged["visits"] * 100
    merged.to_json("conversion.json")
    return merged

# ==============================================================
# 7. ads.json (tu Fase 4, adaptada a las columnas renombradas)
# ==============================================================
def create_ads_dataframe(conversion_df, ads_df, date_start, date_end):
    date_start = pd.to_datetime(date_start).normalize()
    date_end = pd.to_datetime(date_end).normalize()

    conv = conversion_df.copy()
    conv["date_group"] = pd.to_datetime(conv["date_group"]).dt.normalize()
    conv_daily = conv.groupby("date_group", as_index=False).agg(
        visits=("visits", "sum"),
        registrations=("registrations", "sum"),
    )

    ads = ads_df.copy()
    ads["date_group"] = pd.to_datetime(ads["date"]).dt.normalize()
    ads = ads[(ads["date_group"] >= date_start) & (ads["date_group"] <= date_end)]

    if ads.empty:
        ads_daily = pd.DataFrame(columns=["date_group", "cost", "campaign"])
    else:
        ads_daily = ads.groupby("date_group", as_index=False).agg(
            cost=("cost", "sum"),
            campaign=("campaign", "first"),
        )

    result = pd.merge(conv_daily, ads_daily, on="date_group", how="left")
    result["cost"] = result["cost"].fillna(0)
    result["campaign"] = result["campaign"].fillna("none")
    result = result.rename(columns={"campaign": "utm_campaign"})
    result = result.sort_values("date_group").reset_index(drop=True)

    result.to_json("./ads.json", orient="records", date_format="iso")
    return result

# ==============================================================
# 8. PERÍODOS CONTINUOS DE CAMPAÑA
# ==============================================================
def get_continuous_campaign_periods(df, date_group):
    day = {"day": 1, "week": 7, "month": 30, "quarter": 180}
    periods = []

    for campaign in df["full_ad"].unique():
        cdata = df[df["full_ad"] == campaign].copy()
        cdata["date_group"] = pd.to_datetime(cdata["date_group"])
        cdata = cdata.sort_values("date_group")
        cdata["gap"] = cdata["date_group"].diff().dt.days > day[date_group]
        cdata["period"] = cdata["gap"].cumsum()
        cp = cdata.groupby("period").agg(
            start=("date_group", "min"), end=("date_group", "max")
        )
        cp["campaign"] = campaign
        periods.append(cp.reset_index(drop=True))

    return pd.concat(periods, ignore_index=True) if periods else pd.DataFrame()

# ==============================================================
# 9. CARGA DE DATOS
# ==============================================================
def load_data(date_start=DATE_BEGIN, date_end=DATE_END):
    resp = requests.get(f"{API_URL}/registrations",
                        params={"begin": date_start, "end": date_end}, timeout=30)
    resp.raise_for_status()
    regs = pd.DataFrame(resp.json())
    regs.rename(columns={"datetime": "registration_dt"}, inplace=True)

    resp = requests.get(f"{API_URL}/visits",
                        params={"begin": date_start, "end": date_end}, timeout=30)
    resp.raise_for_status()
    visits = pd.DataFrame(resp.json())
    visits.rename(columns={"visit_id": "anonymous_id", "datetime": "visit_dt"}, inplace=True)

    ads = pd.read_csv("./ads.csv")
    ads.rename(columns={"utm_source": "source",
                        "utm_medium": "medium",
                        "utm_campaign": "campaign"}, inplace=True)

    return (clean_registration_data(regs),
            clean_visit_data(visits),
            clean_ads_data(ads))

# ==============================================================
# 10. VISUALIZACIONES
# ==============================================================
def visualize_aggregated_registration_data(data):
    total = data.groupby("date_group").registrations.sum().reset_index()
    plt.figure(figsize=(14, 7))
    sns.barplot(data=total, x="date_group", y="registrations", color="skyblue")
    for i, row in total.iterrows():
        plt.text(i, row.registrations, round(row.registrations, 2), ha="center")
    plt.title("Total Daily Registrations")
    plt.xticks(rotation=45); plt.tight_layout()
    plt.savefig("./charts/total_registrations.png"); plt.show()

    data.pivot_table(index="date_group", columns="platform",
                     values="registrations", fill_value=0) \
        .plot(kind="bar", stacked=True, figsize=(14, 7))
    plt.title("Daily Registrations by Platform (Stacked)")
    plt.xticks(rotation=45); plt.tight_layout()
    plt.savefig("./charts/total_registrations_by_platform.png"); plt.show()

    data.pivot_table(index="date_group", columns="registration_type",
                     values="registrations", fill_value=0) \
        .plot(kind="bar", stacked=True, figsize=(14, 7))
    plt.title("Daily Registrations by Registration Type (Stacked)")
    plt.xticks(rotation=45); plt.tight_layout()
    plt.savefig("./charts/total_registrations_by_type.png"); plt.show()

    platform_pie = data.groupby("platform").registrations.sum()
    type_pie = data.groupby("registration_type").registrations.sum()
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].pie(platform_pie, labels=platform_pie.index, autopct="%1.1f%%", startangle=140)
    ax[0].set_title("Registrations by Platform")
    ax[1].pie(type_pie, labels=type_pie.index, autopct="%1.1f%%", startangle=140)
    ax[1].set_title("Registrations by Type")
    plt.tight_layout()
    plt.savefig("./charts/total_registrations_by_type_pie.png"); plt.show()


def visualize_aggregated_visits_data(data):
    total = data.groupby("date_group").visits.sum().reset_index()
    plt.figure(figsize=(14, 7))
    sns.barplot(data=total, x="date_group", y="visits", color="skyblue")
    for i, row in total.iterrows():
        plt.text(i, row.visits, round(row.visits, 2), ha="center")
    plt.title("Total Visits")
    plt.xticks(rotation=45); plt.tight_layout()
    plt.savefig("./charts/total_visits.png"); plt.show()

    data.pivot_table(index="date_group", columns="platform",
                     values="visits", fill_value=0) \
        .plot(kind="bar", stacked=True, figsize=(14, 7))
    plt.title("Visits by Platform (Stacked)")
    plt.xticks(rotation=45); plt.tight_layout()
    plt.savefig("./charts/total_visits_by_platform.png"); plt.show()


def plot_conversion_graphs(df):
    overall = df.groupby("date_group")[["visits", "registrations"]].sum()
    overall["conversion"] = overall["registrations"] / overall["visits"] * 100

    plt.figure(figsize=(10, 6))
    plt.plot(overall.index, overall["conversion"], marker="o", label="Overall Conversion")
    for x, y in zip(overall.index, overall["conversion"]):
        plt.text(x, y, f"{y:.0f}%", ha="center", va="bottom")
    plt.xlabel("Date"); plt.ylabel("Conversion (%)"); plt.title("Overall Conversion")
    plt.legend(); plt.xticks(rotation=45); plt.tight_layout()
    plt.savefig("./charts/conversion.png"); plt.show()

    platforms = df["platform"].unique()
    plt.figure(figsize=(10, 10))
    for i, platform in enumerate(platforms, 1):
        plt.subplot(len(platforms), 1, i)
        pdata = df[df["platform"] == platform]
        sns.lineplot(x="date_group", y="conversion", data=pdata, marker="o", label=platform)
        for x, y in pdata[["date_group", "conversion"]].values:
            plt.text(x, y, f"{y:.0f}%", ha="center", va="bottom")
        plt.title(f"Conversion {platform}")
        plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("./charts/conversion_by_platforms.png"); plt.show()


def visualize_aggregated_ads_data(df, date_group):
    x = df["date_group"].astype(str)
    y = df["cost"]
    plt.figure(figsize=(12, 6))
    plt.plot(x, y, marker="o", color="b")
    for i, label in enumerate(y):
        plt.text(x.iloc[i], y.iloc[i], f"{label} RUB", ha="right", va="bottom")
    plt.xlabel("Date"); plt.ylabel("Cost (RUB)")
    plt.title(f"Aggregated Ad Campaign Costs (by {date_group})")
    plt.xticks(rotation=45); plt.tight_layout()
    plt.savefig("./charts/ads_cost.png"); plt.show()


def visualize_combined_data(agg_visits, agg_regs, df_ads_periods):
    if df_ads_periods.empty:
        return
    colors = dict(zip(df_ads_periods["campaign"].unique(), mcolors.TABLEAU_COLORS))

    plt.figure(figsize=(12, 12))
    for pos, (data, col, color, label, ylabel, title) in enumerate([
        (agg_visits, "visits", "black", "Visits", "Unique Visits",
         "Visits during marketing active days"),
        (agg_regs, "registrations", "green", "Registrations", "Unique Users",
         "Registrations during marketing active days"),
    ], 1):
        plt.subplot(2, 1, pos)
        total = data.groupby("date_group")[col].sum().reset_index()
        plt.plot(total["date_group"], total[col], color=color, label=label, marker="o")
        plt.axhline(y=total[col].mean(), color="gray", linestyle="--",
                    label=f"Average Number of {label}")
        for _, row in df_ads_periods.iterrows():
            plt.axvspan(row["start"], row["end"], label=row["campaign"],
                        color=colors[row["campaign"]], alpha=0.5)
        plt.title(title); plt.ylabel(ylabel); plt.xticks(rotation=45)
        h, l = plt.gca().get_legend_handles_labels()
        by_label = dict(zip(l, h))
        plt.legend(by_label.values(), by_label.keys())

    plt.tight_layout()
    plt.savefig("./charts/activity_during_marketing_campaign.png"); plt.show()

# ==============================================================
# 11. EJECUCIÓN
# ==============================================================
clean_regs, clean_visits, ads_data_cleaned = load_data(DATE_BEGIN, DATE_END)
date_group = "day"

agg_with_regtype, agg_without_regtype = filter_and_aggregate_registration_data(
    clean_regs, DATE_BEGIN, DATE_END, date_group)
aggregated_visits = filter_and_aggregate_visit_data(
    clean_visits, DATE_BEGIN, DATE_END, date_group)
aggregated_ads_data, df_ads_aggregated_with_ads = filter_and_aggregate_ads_data(
    ads_data_cleaned, DATE_BEGIN, DATE_END, date_group)

print(f"Registros: {len(clean_regs)} | Visitas: {len(clean_visits)} | Publicidad: {len(ads_data_cleaned)}")

if not agg_with_regtype.empty:
    visualize_aggregated_registration_data(agg_with_regtype)

if not aggregated_visits.empty:
    visualize_aggregated_visits_data(aggregated_visits)

if not (agg_without_regtype.empty or aggregated_visits.empty):
    conversion = merge_dataframes_and_calculate_conversion(aggregated_visits, agg_without_regtype)
    plot_conversion_graphs(conversion)
    create_ads_dataframe(conversion, ads_data_cleaned, DATE_BEGIN, DATE_END)

if not aggregated_ads_data.empty:
    visualize_aggregated_ads_data(aggregated_ads_data, date_group)
    df_ads_periods = get_continuous_campaign_periods(df_ads_aggregated_with_ads, date_group)
    visualize_combined_data(aggregated_visits, agg_without_regtype, df_ads_periods)
